Required Packages

In [20]:
# Run this cell to install the required packages
!pip install google-genai google-adk

Set up environment

In [21]:
import os
import json
import asyncio
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types
from google import genai

os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = "qwiklabs-gcp-04-23b9cb055c5f"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"


Agents

In [22]:
client = genai.Client()

# ---------------------------------------------------------
# THE AGENTS (Now defined as explicit Python Tools)
# ---------------------------------------------------------

def intel_research_agent(topic: str, red_cell_feedback: str = "") -> str:
    """Drafts or revises a geopolitical scenario. Pass the topic, and any feedback from the red cell."""
    prompt = f"You are the lead scenario designer. Draft a realistic geopolitical scenario. Topic: {topic}."
    if red_cell_feedback:
        prompt += f"\n\nRevise the scenario to address this Red Cell feedback: {red_cell_feedback}"

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            # Grounding the intel agent with Google Search directly!
            tools=[types.Tool(google_search=types.GoogleSearch())],
            temperature=0.2
        )
    )
    return response.text

def red_cell_reviewer_agent(scenario_text: str) -> str:
    """Critiques the scenario for realism. Returns 'APPROVED' if good, or lists required fixes."""
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f"Critique this scenario from an adversarial perspective. Identify flaws. If it is solid, reply with EXACTLY the word 'APPROVED'. If not, list the required fixes.\n\nScenario:\n{scenario_text}",
        config=types.GenerateContentConfig(temperature=0.2)
    )
    return response.text

def scenario_planner_agent(approved_scenario: str) -> str:
    """Builds the base chronological Master Scenario Events List (MSEL) timeline."""
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f"Take this approved scenario and create a chronological timeline of 3 to 5 major events or triggers:\n\n{approved_scenario}",
        config=types.GenerateContentConfig(temperature=0.2)
    )
    return response.text

def red_cell_player_agent(timeline: str) -> str:
    """Adds Red Force actions to an existing timeline."""
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f"Take this timeline and append how the adversary (Red Force) will react or initiate actions at each event:\n\n{timeline}",
        config=types.GenerateContentConfig(temperature=0.2)
    )
    return response.text

def blue_cell_player_agent(timeline: str) -> str:
    """Adds Blue Force actions to an existing timeline."""
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f"Take this timeline and append how the US/Allied forces (Blue Force) will counter or react at each event:\n\n{timeline}",
        config=types.GenerateContentConfig(temperature=0.2)
    )
    return response.text

def public_affairs_agent(final_timeline: str) -> str:
    """Writes news articles based on a timeline."""
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f"Take this finalized timeline and write a short, realistic news article or intelligence wire for each major event:\n\n{final_timeline}",
        config=types.GenerateContentConfig(temperature=0.2)
    )
    return response.text

# ---------------------------------------------------------
# THE SUPERVISOR (Updated with Anti-Looping Mandate)
# ---------------------------------------------------------
exercise_director = Agent(
    name="exercise_director",
    model="gemini-2.5-pro",
    description="Orchestrates the entire scenario design pipeline using data tools.",
    instruction="""
    You are the Exercise Director.

    AUTONOMOUS EXECUTION MANDATE:
    - Take the user's initial prompt and autonomously drive the entire pipeline to completion.
    - You MUST pass the FULL text output from one tool directly into the input argument of the next tool.

    You MUST execute this exact pipeline sequentially:

    PHASE 1: THE CRITIQUE LOOP
    1. Call `intel_research_agent` with the user's topic to draft the initial scenario.
    2. Pass the FULL scenario text to `red_cell_reviewer_agent`.
    3. If the reviewer returns issues, call `intel_research_agent` again, passing both the topic AND the reviewer's feedback.
    4. Repeat steps 2 and 3 until the reviewer replies 'APPROVED', OR you reach a MAXIMUM of 3 revision loops. If you hit 3 loops, force the pipeline forward using the latest draft.

    PHASE 2: TIMELINE ASSEMBLY
    5. Pass the final scenario text to `scenario_planner_agent` to get the base timeline.
    6. Pass that base timeline text to `red_cell_player_agent` to add Red actions.
    7. Pass the updated timeline text to `blue_cell_player_agent` to add Blue actions.

    PHASE 3: MEDIA INJECTS
    8. Pass the fully combined timeline text to `public_affairs_agent` to generate the news articles.

    Present ALL final results (The Scenario, The Combined Timeline, and The Media Injects) to the user in a beautifully formatted response only after Phase 3 is completely finished.
    """,
    tools=[
        intel_research_agent, red_cell_reviewer_agent,
        scenario_planner_agent, red_cell_player_agent,
        blue_cell_player_agent, public_affairs_agent
    ],
    generate_content_config=types.GenerateContentConfig(
        automatic_function_calling=types.AutomaticFunctionCallingConfig(
            maximum_remote_calls=25
        ),
        temperature=0.2
    )
)

print(f"Tool-Based Pipeline initialized. Director '{exercise_director.name}' is ready with anti-looping protections.")

Tool-Based Pipeline initialized. Director 'exercise_director' is ready with anti-looping protections.


Chat Loop

In [23]:
runner = InMemoryRunner(
    agent=exercise_director,
    app_name="navy_exercise_lab"
)

session_id = "exercise_planning_v4"
user_id = "planner"

await runner.session_service.create_session(
    app_name="navy_exercise_lab",
    user_id=user_id,
    session_id=session_id
)

print("Exercise Director: Ready with explicit data handoffs! (type 'quit' to exit)")

async def chat_loop():
    while True:
        user_input = input("\nYou: ")
        if user_input.lower() in ['quit', 'exit']:
            break

        message_content = types.Content(
            role="user",
            parts=[types.Part.from_text(text=user_input)]
        )

        final_response_text = ""

        print("\n============================================================")
        print("                 INITIATING AGENT PIPELINE                  ")
        print("============================================================")

        async for event in runner.run_async(
            user_id=user_id,
            session_id=session_id,
            new_message=message_content
        ):
            if event.content and event.content.parts:
                for part in event.content.parts:

                    # 1. TRACE: Outgoing task delegation
                    if hasattr(part, 'function_call') and part.function_call:
                        name = part.function_call.name
                        # Safely convert the args to a readable JSON string
                        try:
                            args_dict = dict(part.function_call.args) if part.function_call.args else {}
                            args_str = json.dumps(args_dict, indent=2)
                        except Exception:
                            args_str = str(part.function_call.args)

                        print(f"\n👉 [DIRECTOR ➔ {name.upper()}]")
                        print(args_str)
                        print("-" * 60)

                    # 2. TRACE: Incoming agent results
                    elif hasattr(part, 'function_response') and part.function_response:
                        name = part.function_response.name
                        try:
                            resp_dict = dict(part.function_response.response) if part.function_response.response else {}
                            resp_str = json.dumps(resp_dict, indent=2)
                        except Exception:
                            resp_str = str(part.function_response.response)

                        print(f"\n👈 [{name.upper()} ➔ DIRECTOR]")
                        print(resp_str)
                        print("-" * 60)

            if event.is_final_response():
                if event.content and event.content.parts:
                    if hasattr(event.content.parts[0], 'text') and event.content.parts[0].text:
                        final_response_text = event.content.parts[0].text
                        print("\n============================================================")
                        print("                     PIPELINE COMPLETE                      ")
                        print("============================================================")
                        print(f"\nExercise Director Final Output:\n{final_response_text}")

await chat_loop()

Exercise Director: Ready with explicit data handoffs! (type 'quit' to exit)

                 INITIATING AGENT PIPELINE                  

👉 [DIRECTOR ➔ INTEL_RESEARCH_AGENT]
{
  "topic": "a blockade of South Korea by North Korea with support from China"
}
------------------------------------------------------------

👈 [INTEL_RESEARCH_AGENT ➔ DIRECTOR]
{
  "result": "## Geopolitical Scenario: \"Crimson Tide\" - A Blockade of South Korea\n\n**Scenario Title:** Crimson Tide\n**Date of Scenario Commencement:** Early 2027\n**Primary Actors:** Democratic People's Republic of Korea (DPRK), People's Republic of China (PRC), Republic of Korea (ROK), United States of America (USA), Japan.\n\n---\n\n### 1. Background and Trigger\n\nBy late 2026, global geopolitical tensions remain high, with ongoing proxy conflicts and economic realignments. The DPRK, under Kim Jong Un, has significantly advanced its tactical nuclear weapons program and missile capabilities, conducting a series of increasingly p